# Python or Rust: which implementation to use

This tool ships twice, in Python and in Rust. Both answer to the same conformance corpus, so on every input that corpus covers they emit identical bytes. **Switching between them changes cost, never behavior** — which is the only reason it is safe to choose on cost alone.

This notebook measures that cost. The short version is that there is no single "how much faster": startup, per-file work, and raw transform throughput each differ by a different factor, and which one you feel depends entirely on how you invoke the tool.

If you want the recommendation without the measurements, skip to the last section.

In [1]:
import platform
import statistics
import subprocess
import sys
import time
from pathlib import Path

REPO = Path(
    subprocess.run(
        ['git', 'rev-parse', '--show-toplevel'],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
)
PYTHON_CLI = REPO / '.venv/bin/unwrap-markdown-prose-py'
RUST_CLI = REPO / 'target/release/unwrap-markdown-prose-rs'

for path in (PYTHON_CLI, RUST_CLI):
    if not path.exists():
        raise SystemExit(f'missing {path}: run `uv sync` and `cargo build --release`')

rustc = subprocess.run(
    ['rustc', '--version'], capture_output=True, text=True, check=False
).stdout.strip()
revision = subprocess.run(
    ['git', 'rev-parse', '--short', 'HEAD'],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()

print(f'machine   {platform.machine()}  {platform.system()} {platform.release()}')
print(f'python    {platform.python_version()}')
print(f'rust      {rustc}')
print(f'revision  {revision}')
print(f'binary    {RUST_CLI.stat().st_size / 1024:.0f} KB')

machine   arm64  Darwin 23.5.0
python    3.10.18
rust      rustc 1.86.0 (05f9846f8 2025-03-31)
revision  83136d6
binary    344 KB


## Method

Each measurement times a whole process invocation, because a process invocation is what a hook or a CI step actually pays for. Three warm-up runs are discarded and thirty are kept.

Both the **minimum** and the **median** are reported. The minimum is the honest figure here: it is the run least disturbed by scheduling, and process startup has no legitimate source of variance that would make a slower run more representative. The median sits beside it so that a large gap between the two is visible rather than hidden — that gap is the signal that the machine was busy and the numbers should be re-taken.

In [2]:
WARMUP, REPS = 3, 30


def measure(argv):
    for _ in range(WARMUP):
        subprocess.run(argv, capture_output=True, check=False)
    samples = []
    for _ in range(REPS):
        started = time.perf_counter()
        subprocess.run(argv, capture_output=True, check=False)
        samples.append((time.perf_counter() - started) * 1000)
    return {'min': min(samples), 'median': statistics.median(samples)}


scratch = Path('/tmp/markdown-prose-bench')
scratch.mkdir(exist_ok=True)

# One hard-wrapped paragraph is the unit of work this tool exists to undo.
PARAGRAPH = 'A paragraph that has been hard\nwrapped across three\nseparate lines.\n\n'
(scratch / 'tiny.md').write_text(PARAGRAPH)

tracked = subprocess.run(
    ['git', 'ls-files', '*.md'], cwd=REPO, capture_output=True, text=True, check=True
).stdout.split()
(scratch / 'tracked.txt').write_text(
    '\n'.join(str(REPO / name) for name in tracked) + '\n'
)

# A synthetic tree, so file count varies independently of this repository.
bulk = scratch / 'bulk'
bulk.mkdir(exist_ok=True)
for index in range(2000):
    (bulk / f'{index}.md').write_text(PARAGRAPH * 3)

# One large document, which amortizes startup away and leaves throughput.
large = scratch / 'large.md'
large.write_text(PARAGRAPH * 60000)

print(f'{len(tracked)} tracked Markdown files')
print(f'large document: {large.stat().st_size / 1e6:.1f} MB')

218 tracked Markdown files
large document: 4.1 MB


In [3]:
def file_list(count):
    listing = scratch / f'bulk-{count}.txt'
    listing.write_text(
        '\n'.join(str(bulk / f'{index}.md') for index in range(count)) + '\n'
    )
    return ['--files-from', str(listing)]


scenarios = [
    ('startup floor: 1 file', [str(scratch / 'tiny.md')]),
    ('a typical commit: 5 files', file_list(5)),
    (
        f'this repository: {len(tracked)} files',
        ['--files-from', str(scratch / 'tracked.txt')],
    ),
    ('a large repository: 2000 files', file_list(2000)),
    (f'one {large.stat().st_size / 1e6:.0f} MB document', [str(large)]),
]

results = []
for label, args in scenarios:
    results.append(
        {
            'scenario': label,
            'python': measure([str(PYTHON_CLI), *args, '--json']),
            'rust': measure([str(RUST_CLI), *args, '--json']),
        }
    )

print(f'| {"scenario":<30} | {"Python":>9} | {"Rust":>9} | {"ratio":>6} |')
print(f'| {"-" * 30} | {"-" * 9} | {"-" * 9} | {"-" * 6} |')
for row in results:
    python_ms, rust_ms = row['python']['min'], row['rust']['min']
    print(
        f'| {row["scenario"]:<30} | {python_ms:>7.1f}ms | '
        f'{rust_ms:>7.1f}ms | {python_ms / rust_ms:>5.1f}x |'
    )

| scenario                       |    Python |      Rust |  ratio |
| ------------------------------ | --------- | --------- | ------ |
| startup floor: 1 file          |    28.5ms |     1.6ms |  18.0x |
| a typical commit: 5 files      |    28.8ms |     1.6ms |  17.4x |
| this repository: 218 files     |    63.9ms |     4.7ms |  13.7x |
| a large repository: 2000 files |   202.1ms |    33.8ms |   6.0x |
| one 4 MB document              |  1060.8ms |    69.9ms |  15.2x |


## Where the time actually goes

The table above mixes two effects, and separating them is what makes a per-channel recommendation possible at all. The next cell measures the floor directly: a bare interpreter doing nothing, the same interpreter after importing this tool's module, and then each command-line entry point on a single file.

The gap between the first two is what importing the tool costs. The gap between the last two is what a hook invocation saves.

In [4]:
floor = {
    'bare interpreter': measure([sys.executable, '-c', 'pass']),
    'interpreter + this module': measure(
        [sys.executable, '-c', 'import markdown_prose_hooks.unwrap']
    ),
    'Python CLI, one file': measure(
        [str(PYTHON_CLI), str(scratch / 'tiny.md'), '--json']
    ),
    'Rust CLI, one file': measure([str(RUST_CLI), str(scratch / 'tiny.md'), '--json']),
}
for label, timing in floor.items():
    print(
        f'{label:<28} {timing["min"]:>6.1f} ms   (median {timing["median"]:>6.1f} ms)'
    )

saving = floor['Python CLI, one file']['min'] - floor['Rust CLI, one file']['min']
print(f'\nper-invocation saving  {saving:>6.1f} ms')
print(f'at 100 commits a day   {saving * 100 / 1000:>6.1f} s/day')

bare interpreter               12.8 ms   (median   13.0 ms)
interpreter + this module      25.2 ms   (median   26.5 ms)
Python CLI, one file           26.6 ms   (median   27.5 ms)
Rust CLI, one file              1.5 ms   (median    1.6 ms)

per-invocation saving    25.2 ms
at 100 commits a day      2.5 s/day


## Throughput, with startup removed

One invocation over a growing number of files separates the constant cost from the marginal one. The intercept is startup; the slope is throughput.

In [5]:
import matplotlib.pyplot as plt

counts = [1, 10, 50, 100, 250, 500, 1000, 2000]
curve = {'Python': [], 'Rust': []}
for count in counts:
    args = file_list(count)
    curve['Python'].append(measure([str(PYTHON_CLI), *args, '--json'])['min'])
    curve['Rust'].append(measure([str(RUST_CLI), *args, '--json'])['min'])

figure, axes = plt.subplots(figsize=(8, 4.5))
for name, series in curve.items():
    axes.plot(counts, series, marker='o', label=name)
axes.set_xlabel('files in one invocation')
axes.set_ylabel('wall clock (ms)')
axes.set_title('Startup is the intercept; throughput is the slope')
axes.legend()
axes.grid(alpha=0.3)
figure.tight_layout()
# Written beside the notebook rather than embedded, and SVG rather than PNG.
# Embedding makes a base64 blob that gets spell-checked like prose, and
# codespell duly read one as a typo. PNG would be worse still: this
# repository's .gitattributes routes `*.png` through Git LFS, so a chart
# would commit as a pointer and show as a broken image to anyone cloning
# without git-lfs. SVG is text, so no rule touches it.
chart = REPO / 'docs/benchmarks.svg'
figure.savefig(chart, dpi=144, bbox_inches='tight')
plt.close(figure)
print(f'chart written to {chart.relative_to(REPO)}')

for name, series in curve.items():
    marginal = (series[-1] - series[0]) / (counts[-1] - counts[0])
    print(
        f'{name:<8} intercept {series[0]:>6.1f} ms   '
        f'marginal {marginal * 1000:>5.1f} us/file'
    )

chart written to docs/benchmarks.svg
Python   intercept   26.7 ms   marginal  89.0 us/file
Rust     intercept    1.7 ms   marginal  16.2 us/file


![Startup is the intercept; throughput is the slope](benchmarks.svg)


## How to choose

Three different multipliers come out of the same tool, and they land on different people:

- **process startup — roughly 19x.** Paid once per invocation. This is the whole cost for a `pre-commit` hook over a handful of files.
- **marginal cost per file — roughly 5x.** The low one, because it is dominated by opening, reading and writing a file, which is a syscall in either language.
- **pure transform, one 4 MB document — roughly 15x.** Strip the per-file input and output away and the transform itself is fifteen times faster.

Exact figures are in the cell outputs above rather than repeated here, because a number written into prose stops matching the moment anybody re-runs the notebook.

### The recommendation

| how you run it | use | why |
| -- | -- | -- |
| `pre-commit`, ordinary repository | **`-py`** | `pre-commit` is itself a Python application, so the interpreter is already there and the hook installs in seconds. The Rust hook builds from source and needs `cargo`. |
| `pre-commit`, monorepo or slow `--all-files` | **`-rs`** | The per-file multiplier is what a large sweep feels, and a from-source build is paid once. |
| GitHub Action | **nothing to choose** | The runner is provisioned fresh every job, so a prebuilt binary of a few hundred kilobytes installs faster than either toolchain *and* runs faster afterwards. |
| A script or CI step with no Python | **`-rs`** | A static binary with no runtime to provision. |
| Command line, Python already present | **`-py`** | `pipx install markdown-prose-hooks` and done. |

### Is the difference worth caring about?

For most repositories, honestly, no. Twenty-five milliseconds per commit against a hundred commits a day is about two and a half seconds. It compounds across a team, and agent-driven work commits far more often than human-driven work does, but it is not a reason to add a toolchain you do not otherwise want.

Where it stops being marginal is a sweep over thousands of files — `--all-files` on a monorepo, or a CI job that lints the whole tree — and the GitHub Action, where the Rust costs nothing to adopt because you never install it yourself.